# ML004 第1版输出恢复与提交校准
源版本：Kaggle Version 1，scriptVersionId=364512772。只复用已保存的20张OOF和100张测试概率图。
输入须包含 ML004 第1版输出及比赛 2026task-4。此Notebook不包含模型训练或重新推理入口。
先重现旧版指标和CSV，再校准血管导出；实际榜分需要提交后确认。


In [ ]:
from __future__ import annotations
import ast
import copy
import gc
import hashlib
import inspect
import json
import math
import os
from pathlib import Path
import random
import re
import tempfile
import time

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

VERSION = 'retina-task4-1.2-audited-export'
CFG = dict(
    DATA_ROOT='/kaggle/input/competitions/2026task-4/segmentation',
    # 找不到目录时会在 /kaggle/input 内查找；通常不用修改。
    TRAIN_IMAGE_DIR=None, TRAIN_MASK_DIR=None,
    SAMPLE_PATH=None, CONVERTER_PATH=None,
    # 仅在自动匹配报歧义时填写：{'1': '01_test.tif', ...}
    TEST_ID_MAP={},
    LABEL_VESSEL_VALUE='auto',  # auto / 0 / 255；这里只影响训练标签读取
    RUN_NAME='retina_5fold_v3', OUTPUT_ROOT='/kaggle/working',
    REUSE_FROM=None,  # 可填旧结果中包含 manifest.json 的目录
    SEED=2026, N_FOLDS=5, EPOCHS=70, STEPS_PER_EPOCH=64,
    BATCH_SIZE=12, PATCH=256, WIDTH=24, NUM_WORKERS=2,
    LR=0.001, MIN_LR_RATIO=0.03, WARMUP_EPOCHS=3,
    WEIGHT_DECAY=0.0001, VAL_EVERY=5, PATIENCE=25,
    TILE=256, STRIDE=160, HALO=32, INFER_BATCH=8,
    TTA=8, AMP=True, REQUIRE_GPU=True,
    # 用血管 Dice 选阈值；官方编码区域 Dice 单独报告。
)
EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.gif', '.bmp'}
# 优先使用实际包含所需汉字的字体；环境没有中文字体时自动用英文图标题。
# 不屏蔽其他警告，不需要安装字体或开启网络。
from matplotlib import font_manager
from matplotlib.ft2font import FT2Font

_PLOT_FONT = None
_PLOT_CHARSET = set()


def configure_plot_fonts(allow_chinese=True):
    global _PLOT_FONT, _PLOT_CHARSET
    _PLOT_FONT, _PLOT_CHARSET = None, set()
    needed = {ord(c) for c in '编号：原图血管概率官方白色为编码区域候选最近邻标签叠加'}
    preferred = ['Microsoft YaHei', 'Noto Sans CJK SC', 'Noto Sans CJK JP',
                 'Noto Sans SC', 'Source Han Sans SC', 'SimHei', 'WenQuanYi Micro Hei']
    candidates = sorted(font_manager.fontManager.ttflist,
                        key=lambda entry: (preferred.index(entry.name)
                                           if entry.name in preferred else len(preferred), entry.name))
    seen = set()
    if allow_chinese:
        for entry in candidates:
            if entry.fname in seen:
                continue
            seen.add(entry.fname)
            try:
                charset = set(FT2Font(entry.fname).get_charmap())
            except (OSError, ValueError, RuntimeError):
                continue
            if needed <= charset:
                _PLOT_FONT = font_manager.FontProperties(fname=entry.fname)
                _PLOT_CHARSET = charset
                break
    plt.rcParams['axes.unicode_minus'] = False
    print('绘图字体：', _PLOT_FONT.get_name() if _PLOT_FONT is not None
          else '当前环境没有可用中文字体，图片标题自动改用英文；训练与 CSV 不受影响。')


def set_plot_title(axis, chinese, english):
    text = str(chinese)
    if _PLOT_FONT is not None and all(ord(c) in _PLOT_CHARSET for c in text):
        return axis.set_title(text, fontproperties=_PLOT_FONT)
    # 回退标题只使用 ASCII，避免文件名中的汉字再次触发缺字警告。
    fallback = str(english).encode('ascii', errors='replace').decode('ascii')
    return axis.set_title(fallback, fontproperties=font_manager.FontProperties(family='DejaVu Sans'))


configure_plot_fonts()


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def atomic_json(value, path):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temp, path)

def image_paths(directory):
    return sorted(p for p in Path(directory).rglob('*') if p.suffix.lower() in EXTS)

def resolve_data(cfg):
    root = Path(cfg['DATA_ROOT'])
    if not (root / 'train').is_dir() or not (root / 'test').is_dir():
        choices = sorted({p.parent for p in Path('/kaggle/input').rglob('train')
                          if p.is_dir() and (p.parent / 'test').is_dir()
                          and ('segmentation' in str(p).lower() or '2026task-4' in str(p))})
        if len(choices) != 1:
            raise RuntimeError(f'无法唯一定位 train/test；请填写 DATA_ROOT。候选：{choices}')
        root = choices[0]
    def find_file(name, override):
        if override:
            p = Path(override)
            if not p.is_file():
                raise FileNotFoundError(str(p))
            return p
        for scope in [root, root.parent]:
            hits = sorted(scope.rglob(name))
            if len(hits) == 1:
                return hits[0]
            if len(hits) > 1:
                raise RuntimeError(f'{name} 有多个候选，请在配置中指定：{hits}')
        hits = sorted(p for p in Path('/kaggle/input').rglob(name)
                      if '2026task-4' in str(p) or '2026task4' in str(p))
        if len(hits) != 1:
            raise FileNotFoundError(f'找不到唯一 {name}；请在配置中指定。候选：{hits}')
        return hits[0]
    return root, find_file('sample_submission.csv', cfg['SAMPLE_PATH']), find_file(
        'segmentation_to_csv.py', cfg['CONVERTER_PATH'])

def read_rgb(path):
    with Image.open(path) as im:
        return np.asarray(im.convert('RGB')).copy()

def read_gray(path):
    with Image.open(path) as im:
        return np.asarray(im.convert('L')).copy()

def binary_like(path):
    a = read_rgb(path)
    a = a[::max(1, a.shape[0] // 192), ::max(1, a.shape[1] // 192)]
    gray = a[..., 0].astype(np.float32)
    achromatic = np.mean(np.max(a, axis=2) - np.min(a, axis=2) <= 3) > 0.99
    lo, hi = float(gray.min()), float(gray.max())
    if hi <= lo:
        return achromatic
    extreme = np.mean((gray <= lo + .04 * (hi - lo)) | (gray >= hi - .04 * (hi - lo)))
    return bool(achromatic and extreme > .97)

def numeric_key(value):
    stem = Path(str(value)).stem
    nums = re.findall(r'\d+', stem)
    return str(int(nums[0])) if nums else None

def fov_mask(rgb):
    initial = (rgb.max(axis=2) > 8).astype(np.uint8)
    initial = cv2.morphologyEx(initial, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    contours, _ = cv2.findContours(initial, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError('原图几乎全黑，无法定位眼底区域。')
    result = np.zeros_like(initial)
    cv2.drawContours(result, [max(contours, key=cv2.contourArea)], -1, 1, -1)
    result = cv2.dilate(result, np.ones((5, 5), np.uint8))
    if result.mean() < .1:
        raise ValueError('有效眼底区域不足 10%，请检查是否误把标签当作原图。')
    return result.astype(bool)

def label_to_vessel(raw, fov, value='auto'):
    lo, hi = int(raw.min()), int(raw.max())
    if lo == hi:
        raise ValueError('标签只有一种像素值，不能作为血管分割标签。')
    high = raw > (lo + hi) / 2
    fraction = float(high[fov].mean())
    if value == 'auto':
        vessel_is_high = fraction < .5
    elif value in (0, 255):
        vessel_is_high = value == 255
    else:
        raise ValueError('LABEL_VESSEL_VALUE 只能为 auto、0 或 255。')
    vessel = high if vessel_is_high else ~high
    ratio = float(vessel[fov].mean())
    if not .005 < ratio < .4:
        raise ValueError(f'有效眼底内血管占比异常：{ratio:.3%}；可能选中了 FOV 掩膜或方向不对。')
    outside = np.logical_and(vessel, ~fov).sum() / max(int(vessel.sum()), 1)
    if outside > .02:
        raise ValueError(f'{outside:.2%} 的标签前景在眼底外，请检查标签方向/原图配对。')
    return vessel.astype(np.uint8), ('white' if vessel_is_high else 'black'), ratio

def match_training(root, cfg):
    if cfg['TRAIN_IMAGE_DIR'] or cfg['TRAIN_MASK_DIR']:
        if not (cfg['TRAIN_IMAGE_DIR'] and cfg['TRAIN_MASK_DIR']):
            raise ValueError('手动指定目录时，TRAIN_IMAGE_DIR 与 TRAIN_MASK_DIR 必须同时填写。')
        originals = image_paths(cfg['TRAIN_IMAGE_DIR'])
        labels = image_paths(cfg['TRAIN_MASK_DIR'])
    else:
        paths = image_paths(root / 'train')
        labels, originals = [], []
        for p in paths:
            (labels if binary_like(p) else originals).append(p)
    if len(originals) != 20:
        raise ValueError(f'应识别出 20 张训练原图，实际 {len(originals)}。请检查目录或手动配置。\n'
                         f'原图：{[str(p) for p in originals]}\n标签：{[str(p) for p in labels]}')
    records, used_labels = [], set()
    for p in originals:
        rgb = read_rgb(p)
        fov = fov_mask(rgb)
        same_id = [q for q in labels if q.stem == p.stem or
                   (numeric_key(p) is not None and numeric_key(q) == numeric_key(p))]
        candidates, reasons = [], []
        for q in same_id:
            lower = str(q).lower()
            if any(s in lower for s in ['field_of_view', '/fov/', '_fov.', 'manual2', 'manual_2']):
                continue
            raw = read_gray(q)
            if raw.shape != rgb.shape[:2]:
                reasons.append(f'{q.name}: 尺寸不匹配 {raw.shape}')
                continue
            try:
                vessel, polarity, ratio = label_to_vessel(raw, fov, cfg['LABEL_VESSEL_VALUE'])
                candidates.append((q, vessel, polarity, ratio))
            except ValueError as e:
                reasons.append(f'{q.name}: {e}')
        if len(candidates) != 1:
            raise ValueError(f'{p.name} 无法唯一匹配血管标签；候选 {[str(x[0]) for x in candidates]}；'
                             f'检查结果 {reasons}。请指定准确的训练原图/血管标签目录。')
        q, vessel, polarity, ratio = candidates[0]
        if q in used_labels:
            raise ValueError(f'标签被重复匹配：{q}')
        used_labels.add(q)
        records.append(dict(path=p, mask_path=q, rgb=rgb, y=vessel, fov=fov,
                            polarity=polarity, vessel_ratio=ratio, key=p.stem,
                            image_hash=sha256_file(p), mask_hash=sha256_file(q)))
    # 相同原图不能跨折；这里直接要求检查，避免意外重复造成虚高验证分数。
    if len({r['image_hash'] for r in records}) != len(records):
        raise ValueError('训练集包含字节完全相同的原图，需先核实并按来源分组，不能直接随机分折。')
    return records

def match_test(root, sample_path, cfg):
    sample = pd.read_csv(sample_path, dtype={'Id': str}, keep_default_na=False)
    if list(sample.columns) != ['Id', 'Predicted'] or len(sample) != 20:
        raise ValueError(f'提交模板应为 20 行、Id/Predicted 两列，实际 {sample.shape}, {sample.columns.tolist()}')
    if sample['Id'].duplicated().any():
        raise ValueError('提交模板存在重复 Id。')
    paths = [p for p in image_paths(root / 'test') if not binary_like(p)]
    if len(paths) != 20:
        raise ValueError(f'测试原图数量不是 20，实际为 {len(paths)}；请检查 test 目录。')
    result, used = [], set()
    for sid in sample['Id']:
        override = cfg['TEST_ID_MAP'].get(str(sid))
        if override:
            matches = [p for p in paths if p.name == override or str(p) == override]
        else:
            matches = [p for p in paths if p.stem == Path(sid).stem or p.name == sid]
            if not matches and numeric_key(sid) is not None:
                matches = [p for p in paths if numeric_key(p) == numeric_key(sid)]
        if len(matches) != 1 or matches[0] in used:
            raise ValueError(f'模板 Id={sid!r} 不能唯一对应测试图；候选 {[p.name for p in matches]}。'
                             '请填写 TEST_ID_MAP，禁止用文件排序位置猜编号。')
        p = matches[0]
        used.add(p)
        rgb = read_rgb(p)
        result.append(dict(id=sid, path=p, key=p.stem, rgb=rgb, fov=fov_mask(rgb),
                           image_hash=sha256_file(p)))
    return sample, result

def encode_reference(binary, order):
    pixels = np.asarray(binary, dtype=np.uint8).ravel(order=order)
    padded = np.concatenate(([0], pixels, [0]))
    edges = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    edges[1::2] -= edges[::2]
    return ' '.join(map(str, edges.tolist()))

def normalize_rle(value):
    if value is None:
        return ''
    if isinstance(value, str):
        text = value.strip()
    elif isinstance(value, (list, tuple, np.ndarray)):
        text = ' '.join(str(int(x)) for x in np.asarray(value).ravel())
    else:
        raise TypeError(f'不支持的 RLE 返回值：{type(value)}')
    if not text:
        return ''
    tokens = text.split()
    if any(re.fullmatch(r'\d+', x) is None for x in tokens):
        raise ValueError('RLE 包含非整数内容。')
    return ' '.join(str(int(x)) for x in tokens)

def decode_rle(text, shape, order):
    text = normalize_rle(text)
    result = np.zeros(int(np.prod(shape)), np.uint8)
    if text:
        values = np.array([int(x) for x in text.split()], dtype=np.int64)
        if len(values) % 2:
            raise ValueError('RLE 不是起点/长度配对。')
        starts, lengths = values[::2], values[1::2]
        ends = starts - 1 + lengths
        if np.any(starts < 1) or np.any(lengths < 1) or np.any(ends > result.size):
            raise ValueError('RLE 存在非正数或超出图像范围。')
        if np.any(starts[1:] <= starts[:-1]) or np.any(starts[1:] - 1 < ends[:-1]):
            raise ValueError('RLE 顺序错误或像素重叠。')
        for start, end in zip(starts - 1, ends):
            result[start:end] = 1
    return result.reshape(shape, order=order)

def official_encoded_pixels(gray):
    """与官方 PIL resize + ToTensor + >0 完全相同的二值区域。"""
    gray = np.asarray(gray, dtype=np.uint8)
    if gray.ndim != 2:
        raise ValueError('官方转换器输入必须为二维灰度图。')
    resized = Image.fromarray(gray).resize((512, 512), Image.Resampling.BILINEAR)
    return (np.asarray(resized) > 0).astype(np.uint8)

def probability_512(probability):
    """先缩放连续概率，再二值化，避免二值血管图缩放后被 >0 阈值侵蚀。"""
    probability = np.asarray(probability, dtype=np.float32)
    if probability.ndim != 2 or not np.isfinite(probability).all():
        raise ValueError('概率图尺寸或数值异常。')
    image = Image.fromarray(probability)
    return np.clip(np.asarray(image.resize((512, 512), Image.Resampling.BILINEAR)), 0, 1)

def target_vessel_512(vessel):
    """语义血管标签用最近邻缩放，保证只保留原来的二值类别。"""
    binary = (np.asarray(vessel) > 0).astype(np.uint8)
    if binary.ndim != 2:
        raise ValueError('血管标签必须为二维数组。')
    return np.asarray(Image.fromarray(binary).resize((512, 512), Image.Resampling.NEAREST))

def identify_official_encoder(path):
    """适配用户提供的 turn_to_str(image_list)，不再寻找 rle 命名函数。"""
    import torchvision
    source = Path(path).read_text(encoding='utf-8-sig')
    tree = ast.parse(source)
    selected = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'turn_to_str']
    if len(selected) != 1:
        raise RuntimeError('当前官方文件未找到 turn_to_str(image_list)，请核对是否为本比赛的脚本。')
    env = {'np': np, 'Image': Image, 'torchvision': torchvision}
    # 仅载入官方这个函数；不执行 os.walk、主程序或它的 CSV 写入逻辑。
    exec(compile(ast.Module(body=selected, type_ignores=[]), str(path), 'exec'), env)
    official = env['turn_to_str']
    probes = []
    gray = np.full((37, 53), 255, np.uint8)
    gray[2:18, 3:9] = 0
    gray[24:31, 19:46] = 0
    gray[1, 50] = 0
    probes.append(gray)
    square = np.full((512, 512), 255, np.uint8)
    square[:5, :11] = 0
    square[91:95, 7:240] = 0
    square[-1, -1] = 0
    probes.append(square)
    probes.extend([np.zeros((17, 29), np.uint8), np.full((23, 19), 255, np.uint8)])
    # 灰度值 1 也会被官方脚本编码，不能改成 >127。
    low_gray = np.zeros((512, 512), np.uint8)
    low_gray[2:7, 11:19] = 1
    low_gray[511, 511] = 255
    probes.append(low_gray)
    with tempfile.TemporaryDirectory(prefix='task4_official_check_') as directory:
        paths = []
        for i, probe in enumerate(probes):
            file = Path(directory) / f'{i}.png'
            Image.fromarray(probe).save(file)
            paths.append(str(file))
        outputs = official(paths)
    if not isinstance(outputs, (list, tuple)) or len(outputs) != len(probes):
        raise ValueError('官方 turn_to_str 返回结果数量异常。')
    for probe, value in zip(probes, outputs):
        encoded_pixels = official_encoded_pixels(probe)
        text = normalize_rle(value)
        if text != encode_reference(encoded_pixels, 'C'):
            raise ValueError('官方函数与已提供的 512×512、按行、非零像素约定不一致。')
        if not np.array_equal(decode_rle(text, (512, 512), 'C'), encoded_pixels):
            raise ValueError('官方转换器的逐像素解码校验失败。')
    info = dict(function='turn_to_str', input='PNG path list', order='C',
                shape=[512, 512], encoded_region='nonzero / white background',
                scoring_region_confirmed=False,
                output_png='vessel=0, background=255',
                official_file=str(path), official_sha256=sha256_file(path))
    print('官方 RLE 自检通过：512×512；按行；编码白色像素。注意：这没有验证服务端评分区域。')
    return dict(info=info, function=official)

def enhance(rgb):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(rgb[..., 1])
    return np.concatenate([rgb, clahe[..., None]], axis=2)

def prepare(cfg):
    cfg = copy.deepcopy(cfg)
    if cfg['N_FOLDS'] < 2 or cfg['N_FOLDS'] > 20:
        raise ValueError('N_FOLDS 必须在 2 到 20 之间。')
    if cfg['TILE'] % 16 or cfg['PATCH'] % 16 or not 1 <= cfg['STRIDE'] <= cfg['TILE']:
        raise ValueError('PATCH/TILE 必须是 16 的倍数，且 1 <= STRIDE <= TILE。')
    if cfg['TTA'] not in (1, 4, 8):
        raise ValueError('TTA 仅支持 1、4、8。')
    seed_all(cfg['SEED'])
    cv2.setNumThreads(0)
    torch.backends.cudnn.benchmark = True
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    root, sample_path, converter_path = resolve_data(cfg)
    print('数据目录：', root)
    adapter = identify_official_encoder(converter_path)
    train = match_training(root, cfg)
    sample, test = match_test(root, sample_path, cfg)
    if {r['image_hash'] for r in train} & {r['image_hash'] for r in test}:
        raise ValueError('训练/测试出现字节相同的原图，请先检查数据目录。')
    for r in train + test:
        r['x'] = enhance(r['rgb'])
    rng = np.random.default_rng(cfg['SEED'])
    chunks = np.array_split(rng.permutation(len(train)), cfg['N_FOLDS'])
    fold_ids = np.zeros(len(train), np.int64)
    for i, indices in enumerate(chunks):
        fold_ids[indices] = i
    manifest = dict(version=VERSION, config=cfg,
                    train=[dict(image=r['path'].name, mask=r['mask_path'].name,
                                shape=list(r['y'].shape), polarity=r['polarity'],
                                image_hash=r['image_hash'], mask_hash=r['mask_hash'],
                                fold=int(fold_ids[i])) for i, r in enumerate(train)],
                    test=[dict(id=r['id'], image=r['path'].name, shape=list(r['rgb'].shape[:2]),
                               image_hash=r['image_hash']) for r in test],
                    sample_sha256=sha256_file(sample_path), rle=adapter['info'])
    signature = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    out = Path(cfg['OUTPUT_ROOT']) / cfg['RUN_NAME']
    out.mkdir(parents=True, exist_ok=True)
    config_file = out / 'manifest.json'
    if config_file.exists():
        old = json.loads(config_file.read_text(encoding='utf-8'))
        if old.get('signature') != signature:
            raise ValueError('此 RUN_NAME 下已有不同配置/数据的结果。请改 RUN_NAME，避免混入旧模型。')
    manifest['signature'] = signature
    atomic_json(manifest, config_file)
    rows = [dict(image=r['path'].name, label=r['mask_path'].name,
                 height=r['y'].shape[0], width=r['y'].shape[1],
                 vessel_value=r['polarity'], vessel_ratio=round(r['vessel_ratio'], 5),
                 fold=int(fold_ids[i])) for i, r in enumerate(train)]
    pd.DataFrame(rows).to_csv(out / 'data_audit.csv', index=False)
    pd.DataFrame([dict(Id=r['id'], test_file=r['path'].name) for r in test]).to_csv(
        out / 'test_id_mapping.csv', index=False)
    print(pd.DataFrame(rows).to_string(index=False))
    print('\n测试 Id 对应关系：\n', pd.DataFrame([{'Id': r['id'], 'image': r['path'].name} for r in test]))
    print('\n训练/标签/测试/模板：', len(train), len(train), len(test), len(sample))
    print('设备：', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU（只用于小样本测试）')
    print('输出目录：', out)
    fig, axes = plt.subplots(3, 3, figsize=(12, 11))
    for row, r in enumerate(train[:3]):
        axes[row, 0].imshow(r['rgb'])
        set_plot_title(axes[row, 0], r['path'].name, f'Train image {row + 1}')
        axes[row, 1].imshow(np.where(r['y'], 0, 255), cmap='gray', vmin=0, vmax=255)
        set_plot_title(axes[row, 1], '标签：黑色血管、白色背景', 'Target: black vessel / white background')
        axes[row, 2].imshow(r['rgb'])
        overlay = np.zeros((*r['y'].shape, 4), np.float32)
        overlay[..., 0] = 1
        overlay[..., 3] = r['y'] * .65
        axes[row, 2].imshow(overlay)
        set_plot_title(axes[row, 2], '血管标签叠加', 'Vessel label overlay')
        for ax in axes[row]:
            ax.axis('off')
    fig.tight_layout()
    fig.savefig(out / 'data_preview.png', dpi=130)
    plt.show()
    return dict(cfg=cfg, train=train, test=test, sample=sample, adapter=adapter,
                folds=fold_ids, out=out, signature=signature,
                device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu'))

def read_probability(path, shape):
    value = np.load(path, allow_pickle=False)
    if value.shape != tuple(shape) or not np.isfinite(value).all():
        raise ValueError(f'概率图的尺寸或有限性检查失败：{path}')
    if value.min() < -1e-6 or value.max() > 1 + 1e-6:
        raise ValueError(f'概率图不在 [0, 1] 内：{path}')
    return np.clip(value.astype(np.float32), 0, 1)

def try_reuse_probabilities(ctx):
    """只在图像、标签、编号、分折及预处理一致时复用旧版原尺寸概率图。"""
    configured = ctx['cfg'].get('REUSE_FROM')
    if configured:
        candidates = [Path(configured)]
    else:
        candidates = [Path('/kaggle/working/retina_5fold_v2')]
        input_root = Path('/kaggle/input')
        if input_root.exists():
            candidates += sorted({p.parent for p in input_root.rglob('manifest.json')
                                  if p.parent.name == 'retina_5fold_v2'})
    candidates = list(dict.fromkeys(candidates))
    compatible = []
    for root in candidates:
        if not (root / 'manifest.json').is_file():
            if configured:
                raise FileNotFoundError(f'REUSE_FROM 中没有 manifest.json：{root}')
            continue
        try:
            manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
            if manifest.get('version') != 'retina-task4-1.1-official-512-background':
                raise ValueError('不是本次上传的 v2 Notebook 产生的缓存，无法确认预处理兼容性')
            for key in ['WIDTH', 'N_FOLDS', 'SEED', 'LABEL_VESSEL_VALUE',
                        'TILE', 'STRIDE', 'HALO', 'TTA']:
                if manifest['config'][key] != ctx['cfg'][key]:
                    raise ValueError(f'配置 {key} 与旧缓存不一致')
            if len(manifest['train']) != len(ctx['train']) or len(manifest['test']) != len(ctx['test']):
                raise ValueError('旧缓存的训练/测试数量不一致')
            for i, (old, record) in enumerate(zip(manifest['train'], ctx['train'])):
                for key in ['image_hash', 'mask_hash']:
                    if old[key] != record[key]:
                        raise ValueError(f'训练图像或标签不一致：{record["path"].name}')
                if old['fold'] != int(ctx['folds'][i]) or old['polarity'] != record['polarity']:
                    raise ValueError('旧缓存的分折或标签方向不一致')
            for old, record in zip(manifest['test'], ctx['test']):
                if str(old['id']) != str(record['id']) or old['image_hash'] != record['image_hash']:
                    raise ValueError('旧缓存的测试编号、图片内容或顺序不一致')
            for i, r in enumerate(ctx['train']):
                read_probability(root / f'fold_{ctx["folds"][i]}' / f'oof_{i:02d}.npy', r['y'].shape)
            for fold in range(ctx['cfg']['N_FOLDS']):
                for i, r in enumerate(ctx['test']):
                    read_probability(root / f'fold_{fold}' / f'test_{i:02d}.npy', r['rgb'].shape[:2])
        except (KeyError, ValueError, OSError) as error:
            if configured:
                raise ValueError(f'指定的旧缓存不能安全复用：{root}；{error}') from error
            print(f'未复用 {root}：{error}')
            continue
        compatible.append((root, manifest))
    if len(compatible) > 1:
        raise ValueError('存在多份兼容的旧结果，请用 REUSE_FROM 指定其中一份：'
                         + ', '.join(str(x[0]) for x in compatible))
    if not compatible:
        print('没有完整且兼容的旧概率图，将按当前配置训练。')
        return None
    root, manifest = compatible[0]
    oof = [read_probability(root / f'fold_{ctx["folds"][i]}' / f'oof_{i:02d}.npy', r['y'].shape)
           * r['fov'] for i, r in enumerate(ctx['train'])]
    tests = []
    for i, r in enumerate(ctx['test']):
        p = np.zeros(r['rgb'].shape[:2], np.float32)
        for fold in range(ctx['cfg']['N_FOLDS']):
            p += read_probability(root / f'fold_{fold}' / f'test_{i:02d}.npy', p.shape)
        tests.append(p / ctx['cfg']['N_FOLDS'] * r['fov'])
    ctx['probability_source'] = str(root)
    atomic_json(dict(source=str(root), source_signature=manifest.get('signature'),
                     note='使用原五折模型的概率图；没有按新版配置重新训练。'),
                ctx['out'] / 'reused_cache.json')
    print('已核对全部原图、标签、分折和概率图；跳过训练，复用：', root)
    return oof, tests

def dice_value(prob, target, threshold=.5):
    pred, truth = prob >= threshold, target.astype(bool)
    return float((2 * np.logical_and(pred, truth).sum() + 1e-7) /
                 (pred.sum() + truth.sum() + 1e-7))

def checked_submission(ctx, strings, regions, path):
    """验证的是编码像素本身；通过不等于已确认服务端评分口径。"""
    submission = ctx['sample'].copy()
    submission['Predicted'] = [normalize_rle(s) for s in strings]
    if len(submission) != len(regions):
        raise ValueError('提交数量和掩码数量不一致。')
    submission.to_csv(path, index=False)
    check = pd.read_csv(path, dtype={'Id': str, 'Predicted': str}, keep_default_na=False)
    if check.columns.tolist() != ['Id', 'Predicted'] or check.Id.tolist() != ctx['sample'].Id.tolist():
        raise ValueError('CSV 列名或编号顺序错误。')
    if check.Id.duplicated().any() or len(check) != 20:
        raise ValueError('CSV 行数或唯一性错误。')
    for text, expected in zip(check.Predicted, regions):
        if not np.array_equal(decode_rle(text, (512, 512), 'C'), expected):
            raise ValueError('CSV 写回后解码结果不一致。')
    return path

def calibrate_and_export(ctx, oof, tests):
    if len(oof) != len(ctx['train']) or len(tests) != len(ctx['test']):
        raise ValueError('概率图数量与数据数量不一致。')
    oof512 = [probability_512(p) for p in oof]
    test512 = [probability_512(p) for p in tests]
    # 血管评价用最近邻标签，避免把非零插值后的像素一律当背景而削细血管。
    targets = [target_vessel_512(r['y']) for r in ctx['train']]
    # 另一个参照严格通过真实官方函数转换原尺寸黑血管标签。
    # 它可能与最近邻标签不同，因此分别报告，不能混用两个 Dice。
    target_pngs = ctx['out'] / 'reference_labels'
    target_pngs.mkdir(exist_ok=True)
    target_paths = []
    for i, r in enumerate(ctx['train']):
        path = target_pngs / f'{i:02d}.png'
        Image.fromarray(np.where(r['y'], 0, 255).astype(np.uint8)).save(path)
        target_paths.append(str(path))
    official_targets_rle = ctx['adapter']['function'](target_paths)
    if len(official_targets_rle) != len(targets):
        raise ValueError('官方函数返回的验证标签数量错误。')
    official_targets = [decode_rle(s, (512, 512), 'C') for s in official_targets_rle]
    for encoded_target, r in zip(official_targets, ctx['train']):
        expected = official_encoded_pixels(np.where(r['y'], 0, 255).astype(np.uint8))
        if not np.array_equal(encoded_target, expected):
            raise ValueError('官方标签编码与提供的转换器约定不一致。')
    rows = []
    thresholds = np.round(np.arange(.10, .901, .01), 2)
    for threshold in thresholds:
        predictions = [p >= threshold for p in oof512]
        rows.append(dict(
            threshold=float(threshold),
            vessel_dice=float(np.mean([dice_value(p, y) for p, y in zip(predictions, targets)])),
            official_encoded_dice=float(np.mean([
                dice_value(~p, y) for p, y in zip(predictions, official_targets)])),
            official_vessel_dice=float(np.mean([
                dice_value(p, 1 - y) for p, y in zip(predictions, official_targets)]))))
    table = pd.DataFrame(rows)
    maximum = float(table.vessel_dice.max())
    tied = table[np.isclose(table.vessel_dice, maximum, atol=1e-8, rtol=0)]
    threshold = float(tied.loc[(tied.threshold - .5).abs().idxmin(), 'threshold'])
    per_image = []
    for i, (p, target, official_target, record) in enumerate(
            zip(oof512, targets, official_targets, ctx['train'])):
        pred = p >= threshold
        per_image.append(dict(
            image=record['path'].name, fold=int(ctx['folds'][i]),
            vessel_dice=dice_value(pred, target),
            official_encoded_dice=dice_value(~pred, official_target),
            official_vessel_dice=dice_value(pred, 1 - official_target),
            target_vessel_fraction=float(target.mean()),
            predicted_vessel_fraction=float(pred.mean())))
    scores = pd.DataFrame(per_image)
    metrics = dict(
        oof_vessel_dice_512=float(scores.vessel_dice.mean()),
        oof_official_encoded_dice_512=float(scores.official_encoded_dice.mean()),
        oof_official_vessel_dice_512=float(scores.official_vessel_dice.mean()),
        official_all_white_baseline=float(np.mean([dice_value(np.ones_like(y), y)
                                                 for y in official_targets])),
        threshold=threshold, threshold_selection_metric='vessel_dice_nearest_neighbor_target',
        probability_source=ctx.get('probability_source', '当前版本训练的五折模型'),
        submission_shape=[512, 512], folds=ctx['cfg']['N_FOLDS'], tta=ctx['cfg']['TTA'],
        rle=ctx['adapter']['info'],
        note='OOF 同时参与模型轮次与阈值选择，不是独立测试结果；不能据此承诺榜分。',
        recommended_submission_region='vessel',
        recommended_submission_file='submission_task4_vessels.csv',
        score_convention='按已有线上对照采用血管编码；官方背景编码仅作参考。')
    table.to_csv(ctx['out'] / 'threshold_search.csv', index=False)
    scores.to_csv(ctx['out'] / 'oof_scores.csv', index=False)
    atomic_json(metrics, ctx['out'] / 'metrics.json')
    print('OOF 血管 Dice（最近邻标签）：', metrics['oof_vessel_dice_512'])
    print('OOF 官方编码区域 Dice（白色背景）：', metrics['oof_official_encoded_dice_512'])
    print('OOF 官方转换后黑色血管 Dice：', metrics['oof_official_vessel_dice_512'])
    print('全部预测为白色背景的基线 Dice：', metrics['official_all_white_baseline'])
    print('所选血管阈值：', threshold)
    if threshold in (float(thresholds[0]), float(thresholds[-1])):
        print('注意：最优阈值在搜索边界，请先检查验证图和概率分布。')

    destination = ctx['out'] / 'predicted_masks'
    destination.mkdir(exist_ok=True)
    paths, vessels, backgrounds, audit = [], [], [], []
    used_names = set()
    for record, native_probability, probability in zip(ctx['test'], tests, test512):
        if native_probability.shape != record['rgb'].shape[:2]:
            raise ValueError('原始测试概率图的尺寸错误。')
        vessel = probability >= threshold
        name = re.sub(r'[^A-Za-z0-9_.-]', '_', str(record['id'])) + '.png'
        if name in used_names or name == '.png':
            raise ValueError('测试编号转换为文件名后发生冲突。')
        used_names.add(name)
        path = destination / name
        gray = np.where(vessel, 0, 255).astype(np.uint8)
        Image.fromarray(gray).save(path)
        reread = read_gray(path)
        if reread.shape != (512, 512) or not np.array_equal(reread == 0, vessel):
            raise ValueError('输出 PNG 的尺寸或黑白方向错误。')
        paths.append(str(path))
        vessels.append(vessel)
        backgrounds.append(~vessel)
        audit.append(dict(Id=record['id'], test_image=record['path'].name,
                          png=name, vessel_fraction=float(vessel.mean()),
                          official_encoded_fraction=float((~vessel).mean()),
                          order='C', height=512, width=512))
    official_strings = [normalize_rle(s) for s in ctx['adapter']['function'](paths)]
    if len(official_strings) != len(paths):
        raise ValueError('官方转换器返回数量不一致。')
    for text, background in zip(official_strings, backgrounds):
        if text != encode_reference(background, 'C'):
            raise ValueError('真实官方函数没有按已核实的约定编码白色像素。')
    reference_dir = ctx['out'] / 'diagnostics'
    reference_dir.mkdir(exist_ok=True)
    official_path = checked_submission(ctx, official_strings, backgrounds,
                                      reference_dir / 'background_reference.csv')
    vessel_path = checked_submission(ctx, [encode_reference(v, 'C') for v in vessels],
                                    vessels, ctx['out'] / 'submission_task4_vessels.csv')
    # 兼容以前的下载路径：这些名字现在全部指向同一份血管数据。
    # 官方脚本生成的背景参考只保存在 diagnostics 中。
    for alias_name in ['submission.csv', 'submission_vessel_candidate.csv']:
        (ctx['out'] / alias_name).write_bytes(vessel_path.read_bytes())
    pd.DataFrame(audit).to_csv(ctx['out'] / 'submission_audit.csv', index=False)
    if any(float(v.mean()) > .5 or float(v.mean()) < .001 for v in vessels):
        print('注意：部分预测的血管占比异常，请检查训练和预览，勿仅凭格式通过就提交。')
    print('血管提交已通过编号、边界、顺序与逐像素解码检查。')
    print('本次请上传：', vessel_path)
    print('血管编码平均占比：', float(np.mean([v.mean() for v in vessels])))
    print('提交文件 SHA256：', sha256_file(vessel_path))
    print('官方脚本的背景编码参考已移到 diagnostics 目录。')
    print('当前默认选择依据已有线上对照分数；新文件的实际分数仍以本次提交为准。')

    count = min(3, len(ctx['test']))
    fig, axes = plt.subplots(count, 4, figsize=(15, 4 * count), squeeze=False)
    for i, record in enumerate(ctx['test'][:count]):
        panels = [record['rgb'], test512[i], backgrounds[i], vessels[i]]
        titles = [f'编号 {record["id"]}：原图', '血管概率',
                  '官方 CSV：白色为编码区域', '候选 CSV：白色为编码区域']
        for j, (panel, title) in enumerate(zip(panels, titles)):
            if j == 0:
                axes[i, j].imshow(panel)
            else:
                axes[i, j].imshow(panel, cmap='magma' if j == 1 else 'gray', vmin=0, vmax=1)
            set_plot_title(axes[i, j], title, [f'Test image {record["id"]}',
                      'Vessel probability', 'Official CSV: encoded pixels',
                      'Vessel candidate: encoded pixels'][j])
            axes[i, j].axis('off')
    fig.tight_layout()
    fig.savefig(ctx['out'] / 'prediction_preview.png', dpi=130)
    plt.show()
    return metrics, vessel_path


# 固定使用浏览器中核实过的旧版输出。发现缺失时停止，不训练、不重新推理。
EXPECTED_V1_OOF = 0.7582848355285299
LEGACY_VERSION = 'retina-task4-1.1-official-512-background'
candidate_roots = []
discovery_records = []
# Kaggle 新挂载层级可能包含目录符号链接；记录已访问的实际目录以避免循环。
manifest_paths = []
visited_directories = set()
for folder, subdirectories, filenames in os.walk('/kaggle/input', followlinks=True):
    resolved = os.path.realpath(folder)
    if resolved in visited_directories:
        subdirectories[:] = []
        continue
    visited_directories.add(resolved)
    if 'manifest.json' in filenames:
        manifest_paths.append(Path(folder) / 'manifest.json')
for manifest_path in manifest_paths:
    found = dict(manifest=str(manifest_path))
    discovery_records.append(found)
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        root = manifest_path.parent
        found.update(version=manifest.get('version'),
                     oof_files=len(list(root.glob('fold_*/oof_*.npy'))),
                     test_files=len(list(root.glob('fold_*/test_*.npy'))),
                     best_models=len(list(root.glob('fold_*/best.pt'))))
        if manifest.get('version') != LEGACY_VERSION:
            found['reason'] = '缓存版本标记不符'
            continue
        old_metrics = json.loads((root / 'metrics.json').read_text(encoding='utf-8'))
        found['old_oof'] = old_metrics.get('oof_vessel_dice_512')
        if abs(float(old_metrics['oof_vessel_dice_512']) - EXPECTED_V1_OOF) > 1e-8:
            found['reason'] = '旧版OOF记录与第1版日志不符'
            continue
        if len(manifest['train']) != 20 or len(manifest['test']) != 20:
            continue
        paths = [root / f'fold_{int(r["fold"])}' / f'oof_{i:02d}.npy'
                 for i, r in enumerate(manifest['train'])]
        paths += [root / f'fold_{fold}' / f'test_{i:02d}.npy'
                  for fold in range(5) for i in range(20)]
        paths += [root / f'fold_{fold}' / 'best.pt' for fold in range(5)]
        paths += [root / 'submission.csv']
        if all(p.is_file() for p in paths):
            candidate_roots.append(root)
            found['reason'] = '完整且匹配'
        else:
            found['reason'] = '缺少所需文件'
            found['missing'] = [str(p) for p in paths if not p.is_file()][:8]
    except (ValueError, KeyError, OSError) as error:
        found['reason'] = str(error)
        continue
discovery_path = Path('/kaggle/working/recovery_discovery.json')
discovery_path.write_text(json.dumps(discovery_records, ensure_ascii=False, indent=2), encoding='utf-8')
print('缓存发现记录：', json.dumps(discovery_records, ensure_ascii=False), flush=True)
if len(candidate_roots) != 1:
    raise RuntimeError(
        '没有找到唯一的 ML004 第1版完整输出。请通过 Add Input 挂载 '
        'ML004 的 Version 1（364512772，171个文件），并添加比赛 2026task-4。'
        f'当前匹配目录：{candidate_roots}。本程序不会训练。')
RECOVERED_V1 = candidate_roots[0]
print('确认旧版缓存：', RECOVERED_V1, flush=True)
CFG.update(REQUIRE_GPU=False, REUSE_FROM=str(RECOVERED_V1),
           RUN_NAME='retina_ml004_v1_recovered', NUM_WORKERS=0)
ctx = prepare(CFG)
reused_probabilities = try_reuse_probabilities(ctx)
if reused_probabilities is None:
    raise RuntimeError('旧概率图不能完整复用，停止；不会转入训练或模型推理。')
oof_probabilities, test_probabilities = reused_probabilities

# 先重现旧版本的指标和20张测试掩码，防止拿错版本。
legacy_metrics = json.loads((RECOVERED_V1 / 'metrics.json').read_text(encoding='utf-8'))
legacy_threshold = float(legacy_metrics['threshold'])
legacy_targets = [1 - official_encoded_pixels(np.where(r['y'], 0, 255).astype(np.uint8))
                  for r in ctx['train']]
legacy_predictions = [probability_512(p) >= legacy_threshold for p in oof_probabilities]
legacy_reproduced = float(np.mean([dice_value(p, y) for p, y in zip(legacy_predictions, legacy_targets)]))
if abs(legacy_reproduced - float(legacy_metrics['oof_vessel_dice_512'])) > 1e-5:
    raise ValueError(f'旧版OOF无法重现：{legacy_reproduced}，记录为{legacy_metrics["oof_vessel_dice_512"]}。停止导出。')
legacy_csv = pd.read_csv(RECOVERED_V1 / 'submission.csv', dtype=str, keep_default_na=False)
if legacy_csv.columns.tolist() != ['Id', 'Predicted'] or legacy_csv.Id.tolist() != ctx['sample'].Id.tolist():
    raise ValueError('旧提交的编号或列名不匹配。')
for row, p in zip(legacy_csv.itertuples(index=False), test_probabilities):
    if not np.array_equal(decode_rle(row.Predicted, (512, 512), 'C'),
                          ~(probability_512(p) >= legacy_threshold)):
        raise ValueError('旧提交与缓存测试预测不一致，停止导出。')
recovery_audit = dict(
    kaggle_saved_version=1, script_version_id=364512772,
    source=str(RECOVERED_V1), old_threshold=legacy_threshold,
    old_oof_recorded=float(legacy_metrics['oof_vessel_dice_512']),
    old_oof_reproduced=legacy_reproduced,
    original_submission_sha256=sha256_file(RECOVERED_V1 / 'submission.csv'),
    source_manifest_sha256=sha256_file(RECOVERED_V1 / 'manifest.json'),
    best_model_sha256=[sha256_file(RECOVERED_V1 / f'fold_{fold}' / 'best.pt') for fold in range(5)],
    model_training_executed=False, model_inference_executed=False,
    separate_v3_public_score_user_reported=0.72,
    note='0.72来自另一次v3预测，不是这批恢复概率图的线上成绩。原始模型不在本程序中加载。')
atomic_json(recovery_audit, ctx['out'] / 'recovery_audit.json')
print('旧版OOF及20张旧提交掩码均已重现：', legacy_reproduced, flush=True)

# 重新按最近邻血管标签校准；主CSV只编码血管。
metrics, submission_path = calibrate_and_export(ctx, oof_probabilities, test_probabilities)


"""任务4：追加到已运行的 v3 Notebook 末尾，只校准和导出，不训练。

使用：复制本文件全部内容，在原 Kaggle Notebook 最下面添加一个 Code 单元格，
粘贴后只运行该新单元格。请保留原五折输出，不需要从头运行全部 Notebook。
出现“下载待验证提交 CSV”时，可以下载并记录实际榜分；请同时把最后的摘要发回。

也可单独运行：读取已保存的 v3 概率图。缺少缓存时停止，不启动训练。
训练标签的原始白色区域经官方脚本编码后，才作为该分支的校准目标。
隐藏评分标签的生成方式仍未公开，校准分数不是榜分保证。
"""

from pathlib import Path

import ast

import base64

import csv

import hashlib

import json

import re

import tempfile

import numpy as np

from PIL import Image

TASK4_RUN_DIR = None

TASK4_OUTPUT_DIR = '/kaggle/working/retina_calibration'

def t4_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def t4_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def t4_gray(path):
    with Image.open(path) as im:
        return np.asarray(im.convert('L')).copy()

def t4_probability(value, shape):
    value = np.asarray(value, dtype=np.float32)
    if value.shape != tuple(shape) or not np.isfinite(value).all():
        raise ValueError('概率图尺寸或数值异常，请核对缓存来源。')
    if value.min() < -1e-6 or value.max() > 1 + 1e-6:
        raise ValueError('概率图不在 [0, 1] 内。')
    return np.clip(value, 0, 1)

def t4_prob512(value):
    return np.asarray(Image.fromarray(np.asarray(value, np.float32)).resize(
        (512, 512), Image.Resampling.BILINEAR))

def t4_nonzero512(gray):
    return np.asarray(Image.fromarray(np.asarray(gray, np.uint8)).resize(
        (512, 512), Image.Resampling.BILINEAR)) > 0

def t4_nearest(vessel):
    return np.asarray(Image.fromarray(np.asarray(vessel, np.uint8)).resize(
        (512, 512), Image.Resampling.NEAREST)) > 0

def t4_rle(mask):
    flat = np.asarray(mask, np.uint8).ravel(order='C')
    edges = np.flatnonzero(np.diff(np.r_[0, flat, 0])) + 1
    edges[1::2] -= edges[::2]
    return ' '.join(map(str, edges.tolist()))

def t4_decode(text):
    values = np.asarray(str(text).split(), dtype=np.int64)
    if len(values) % 2:
        raise ValueError('RLE 起点与长度未成对。')
    starts, lengths = values[::2] - 1, values[1::2]
    ends = starts + lengths
    if (np.any(starts < 0) or np.any(lengths <= 0) or
            np.any(ends > 512 * 512) or np.any(starts[1:] < ends[:-1])):
        raise ValueError('RLE 乱序、重叠或越界。')
    changes = np.zeros(512 * 512 + 1, dtype=np.int32)
    np.add.at(changes, starts, 1)
    np.add.at(changes, ends, -1)
    return (np.cumsum(changes[:-1]) > 0).reshape(512, 512)

def t4_dice(prediction, target):
    prediction, target = np.asarray(prediction, bool), np.asarray(target, bool)
    return float((2 * np.logical_and(prediction, target).sum() + 1e-7) /
                 (prediction.sum() + target.sum() + 1e-7))

def t4_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ['Id', 'Predicted']:
            raise ValueError(f'提交列名错误：{path}')
        return list(reader)

def t4_converter(path):
    # 只载入用户提供的官方函数，不执行脚本的主程序。
    import torchvision
    tree = ast.parse(Path(path).read_text(encoding='utf-8-sig'))
    selected = [n for n in tree.body if isinstance(n, ast.FunctionDef)
                and n.name == 'turn_to_str']
    if len(selected) != 1:
        raise ValueError('官方文件中没有唯一的 turn_to_str 函数。')
    scope = dict(np=np, Image=Image, torchvision=torchvision)
    exec(compile(ast.Module(body=selected, type_ignores=[]), str(path), 'exec'), scope)
    return scope['turn_to_str']

def t4_fov(path):
    # 与 v3 的眼底范围处理保持一致。
    import cv2
    with Image.open(path) as im:
        rgb = np.asarray(im.convert('RGB'))
    initial = (rgb.max(axis=2) > 8).astype(np.uint8)
    initial = cv2.morphologyEx(initial, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    contours, _ = cv2.findContours(initial, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError(f'原图没有有效眼底区域：{path}')
    field = np.zeros_like(initial)
    cv2.drawContours(field, [max(contours, key=cv2.contourArea)], -1, 1, -1)
    return cv2.dilate(field, np.ones((5, 5), np.uint8)).astype(bool)

class T4CacheUnavailable(RuntimeError):
    """已有缓存尚未挂载、缺失或无法唯一选择；不会自动启动训练。"""

def t4_cache_status(root):
    """按缓存内容识别版本，目录可以改名或增加挂载层级。"""
    root = Path(root)
    status = dict(root=root, version=None, ready=False, oof=0, test=0, weights=0,
                  metrics=False, missing=[])
    try:
        manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
        status['version'] = manifest.get('version')
        if status['version'] != 'retina-task4-1.2-audited-export':
            return status
        train, test = manifest['train'], manifest['test']
        fold_count = int(manifest['config']['N_FOLDS'])
        if (len(train) != 20 or len(test) != 20 or fold_count != 5 or
                {int(r['fold']) for r in train} != set(range(5))):
            raise ValueError('不是完整的 20 张训练图、20 张测试图及五折记录')
        source = root
        if (root / 'reused_cache.json').is_file():
            saved_source = json.loads((root / 'reused_cache.json').read_text(encoding='utf-8'))
            source = Path(saved_source['source'])
        status['source'] = source
        oof = [source / f'fold_{int(r["fold"])}' / f'oof_{i:02d}.npy'
               for i, r in enumerate(train)]
        predictions = [source / f'fold_{fold}' / f'test_{i:02d}.npy'
                       for fold in range(5) for i in range(20)]
        weights = [source / f'fold_{fold}' / 'best.pt' for fold in range(5)]
        status.update(oof=sum(p.is_file() for p in oof),
                      test=sum(p.is_file() for p in predictions),
                      weights=sum(p.is_file() for p in weights))
        metadata = [root / 'metrics.json', source / 'manifest.json']
        if source == root:
            metadata += [source / f'fold_{fold}' / 'prediction_source.json' for fold in range(5)]
        status['missing'] = [str(p) for p in oof + predictions + weights + metadata if not p.is_file()]
        if (root / 'metrics.json').is_file():
            metrics = json.loads((root / 'metrics.json').read_text(encoding='utf-8'))
            status['metrics'] = all(k in metrics for k in ['threshold', 'oof_vessel_dice_512'])
        status['ready'] = not status['missing'] and status['metrics']
    except (OSError, ValueError, KeyError, TypeError) as error:
        status['error'] = str(error)
    return status

def t4_find_cached_run(run_dir=None, search_roots=None):
    """检查所有已挂载的候选；不根据目录名称或时间猜测使用哪轮预测。"""
    roots = [Path(p) for p in (search_roots if search_roots is not None else
                              ['/kaggle/working', '/kaggle/input', '/kaggle/working.d'])]
    if run_dir is not None:
        supplied = Path(run_dir)
        candidates = [supplied.parent if supplied.name == 'manifest.json' else supplied]
    else:
        candidates = sorted({p.parent.resolve() for base in roots if base.is_dir()
                             for p in base.rglob('manifest.json')})
    statuses = [t4_cache_status(path) for path in candidates]
    compatible = [s for s in statuses if s['version'] == 'retina-task4-1.2-audited-export']
    ready = [s for s in compatible if s['ready']]
    if len(ready) == 1:
        print('找到完整 v3 缓存：', ready[0]['root'])
        return ready[0]['root']
    if len(ready) > 1:
        paths = '\n'.join(f'  {s["root"]}' for s in ready)
        raise T4CacheUnavailable('找到多份完整 v3 输出，不能猜测哪份对应 0.72 分：\n' + paths +
                                '\n请把所需目录填写到 TASK4_RUN_DIR，再运行本单元格。')
    lines = ['没有找到可直接校准的完整 v3 缓存。本次没有训练，也没有修改旧提交。']
    if compatible:
        for status in compatible:
            lines.append(f'{status["root"]}\n  OOF 概率 {status["oof"]}/20，'
                         f'测试概率 {status["test"]}/100，模型 {status["weights"]}/5，'
                         f'评分记录：{"有" if status["metrics"] else "无"}')
            if status.get('error'):
                lines.append('  配置记录异常：' + status['error'])
            elif status.get('missing'):
                lines.append('  缺少文件示例：' + status['missing'][0])
        if any(s['weights'] == 5 and (s['oof'] < 20 or s['test'] < 100) for s in compatible):
            lines.append('存在五折模型但概率图不全：可以另行只做推理；当前校准单元格不会自动推理。')
    else:
        lines.append('已检查目录：' + '、'.join(map(str, roots)) if run_dir is None
                     else '指定目录不是有效的 v3 输出：' + str(run_dir))
        lines.append('尚未发现匹配的 manifest.json；这不能证明其他已保存版本中也没有输出。')
    lines.extend(['请先在当前 Kaggle Notebook 的 Input → Add Input 中，添加原来训练完成的 Notebook 输出。',
                  '需要包含 manifest.json、metrics.json 和 fold_0 等五折目录，然后只重跑本单元格。',
                  '仅有 submission.csv 无法还原概率图；若任何保存版本中都没有模型和概率缓存，才需要重新训练。'])
    raise T4CacheUnavailable('\n'.join(lines))

def t4_cached_inputs(run_dir):
    """内存变量不存在时，只读磁盘缓存；没有任何训练或模型推理入口。"""
    root = t4_find_cached_run(run_dir)
    manifest = t4_json(root / 'manifest.json')
    if manifest['version'] != 'retina-task4-1.2-audited-export':
        raise ValueError('这里只接受已核查的 v3 输出。')
    data_root = Path(manifest['config']['DATA_ROOT'])
    if not (data_root / 'train').is_dir():
        roots = [p.parent for p in Path('/kaggle/input').rglob('train')
                 if p.is_dir() and (p.parent / 'test').is_dir()
                 and ('2026task-4' in str(p) or p.parent.name == 'segmentation')]
        if len(roots) != 1:
            raise RuntimeError('找不到原训练数据，请把本比赛数据添加到 Notebook。')
        data_root = roots[0]

    def locate(name, expected_hash, split):
        matches = [p for p in (data_root / split).rglob(name)
                   if p.is_file() and t4_hash(p) == expected_hash]
        if len(matches) != 1:
            raise ValueError(f'无法按文件名及哈希唯一匹配原始文件：{name}')
        return matches[0]

    train, test = [], []
    for old in manifest['train']:
        image = locate(old['image'], old['image_hash'], 'train')
        mask = locate(old['mask'], old['mask_hash'], 'train')
        raw = t4_gray(mask)
        high = raw > (int(raw.min()) + int(raw.max())) / 2
        if old['polarity'] not in ('white', 'black'):
            raise ValueError('缓存中的标签颜色未知。')
        y = high if old['polarity'] == 'white' else ~high
        train.append(dict(path=image, mask_path=mask, y=y, fov=t4_fov(image),
                          polarity=old['polarity'], mask_hash=old['mask_hash']))
    for old in manifest['test']:
        image = locate(old['image'], old['image_hash'], 'test')
        test.append(dict(path=image, id=str(old['id']), fov=t4_fov(image)))
    source = root
    if (root / 'reused_cache.json').is_file():
        source = Path(t4_json(root / 'reused_cache.json')['source'])
    old_manifest = t4_json(source / 'manifest.json')
    if (old_manifest['train'] != manifest['train'] or
            old_manifest['test'] != manifest['test']):
        raise ValueError('概率缓存的图像、编号或分折与 v3 不一致。')
    folds = np.asarray([r['fold'] for r in manifest['train']], dtype=int)
    fold_count = int(manifest['config']['N_FOLDS'])
    for fold in range(fold_count):
        folder = source / f'fold_{fold}'
        provenance = folder / 'prediction_source.json'
        if provenance.is_file():
            info = t4_json(provenance)
            expected_inference = {k: manifest['config'][k] for k in ['TILE', 'STRIDE', 'HALO', 'TTA']}
            if (info['signature'] != old_manifest['signature'] or
                    info['inference'] != expected_inference or
                    info['model_sha256'] != t4_hash(folder / 'best.pt')):
                raise ValueError('模型或推理配置与概率缓存不一致。')
        elif source == root:
            raise ValueError('v3 概率缓存缺少来源记录，不能安全校准。')

    def probability(split, index, fold, record):
        value = np.load(source / f'fold_{fold}' / f'{split}_{index:02d}.npy', allow_pickle=False)
        return t4_probability(value, record['fov'].shape)

    oof = [probability('oof', i, int(folds[i]), r) * r['fov'] for i, r in enumerate(train)]
    tests = []
    for i, r in enumerate(test):
        summed = np.zeros(r['fov'].shape, dtype=np.float32)
        for fold in range(fold_count):
            summed += probability('test', i, fold, r)
        tests.append(summed / fold_count * r['fov'])
    converter = Path(manifest['rle']['official_file'])
    if not converter.is_file():
        matches = [p for p in data_root.parent.rglob('segmentation_to_csv.py')
                   if t4_hash(p) == manifest['rle']['official_sha256']]
        if len(matches) != 1:
            raise ValueError('找不到原官方转换器。')
        converter = matches[0]
    if t4_hash(converter) != manifest['rle']['official_sha256']:
        raise ValueError('官方转换器内容已变化。')
    return dict(root=root, train=train, test=test, oof=oof, tests=tests, folds=folds,
                official=t4_converter(converter), metrics=t4_json(root / 'metrics.json'),
                source=str(source))

def t4_inputs(scope):
    context = scope.get('ctx')
    if TASK4_RUN_DIR is not None or context is None or any(
            key not in scope for key in ['oof_probabilities', 'test_probabilities']):
        return t4_cached_inputs(TASK4_RUN_DIR)
    root = Path(context['out'])
    saved = t4_json(root / 'manifest.json')
    if saved['signature'] != context['signature']:
        raise ValueError('当前内存上下文与磁盘记录不一致。')
    for record in context['train']:
        if t4_hash(record['mask_path']) != record['mask_hash']:
            raise ValueError('原训练标签已变化。')
    if context['sample'].Id.tolist() != [r['id'] for r in context['test']]:
        raise ValueError('模板编号与测试图顺序不一致。')
    return dict(root=root, train=context['train'], test=context['test'],
                oof=scope['oof_probabilities'], tests=scope['test_probabilities'],
                folds=np.asarray(context['folds'], int), official=context['adapter']['function'],
                metrics=t4_json(root / 'metrics.json'),
                source=context.get('probability_source', str(root)))

def t4_native_binary(native, method, threshold):
    binary = (native >= threshold).astype(np.uint8)
    if '_minarea_' in method:
        minimum = int(method.rsplit('_minarea_', 1)[1])
        count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
        keep = stats[:, cv2.CC_STAT_AREA] >= minimum
        keep[0] = False
        binary = keep[labels].astype(np.uint8)
    return binary

def t4_prediction(native, resized, method, threshold):
    if method == 'probability_then_threshold':
        return resized >= threshold
    if method.startswith('native_mask_then_official_resize'):
        # 在原图尺寸确定血管，再按官方白色像素的缩放规则编码。
        return t4_nonzero512(t4_native_binary(native, method, threshold) * 255)
    raise ValueError('未知的导出流程。')

def t4_select(score_matrix, candidates, indices):
    scores = score_matrix[:, indices].mean(axis=1)
    best = np.flatnonzero(np.isclose(scores, scores.max(), atol=1e-8, rtol=0))
    # 分数相同时优先保持已有的处理方式，再选择较接近 0.5 的阈值。
    return min(best.tolist(), key=lambda i: (
        candidates[i][0] != 'probability_then_threshold', abs(candidates[i][1] - .5)))

def t4_calibrate(data, destination, show_download=True):
    train, test = data['train'], data['test']
    folds = np.asarray(data['folds'], int)
    ids = [str(r['id']) for r in test]
    if (len(train) != 20 or len(test) != 20 or len(set(ids)) != 20 or
            folds.shape != (20,) or set(folds.tolist()) != set(range(5))):
        raise ValueError('需要 20 张训练图、20 个唯一测试编号及完整五折记录。')
    if any(re.fullmatch(r'[A-Za-z0-9_.-]+', sid) is None or sid in {'.', '..'} for sid in ids):
        raise ValueError('测试编号含不适合作为 PNG 文件名的字符，请先核对模板。')
    png_names = [sid + '.png' for sid in ids]
    if len(data['oof']) != 20 or len(data['tests']) != 20:
        raise ValueError('缺少 OOF 或测试集概率图。')
    oof = [t4_probability(p, r['y'].shape) * r['fov'] for p, r in zip(data['oof'], train)]
    tests = [t4_probability(p, r['fov'].shape) * r['fov'] for p, r in zip(data['tests'], test)]
    oof512, test512 = list(map(t4_prob512, oof)), list(map(t4_prob512, tests))
    nearest = [t4_nearest(r['y']) for r in train]
    threshold = float(data['metrics']['threshold'])
    baseline = [p >= threshold for p in oof512]
    baseline_nn = float(np.mean([t4_dice(p, y) for p, y in zip(baseline, nearest)]))
    original_score = float(data['metrics']['oof_vessel_dice_512'])
    if abs(baseline_nn - original_score) > 1e-5:
        raise ValueError(f'无法重现旧 OOF 分数：记录 {original_score:.8f}，现在 {baseline_nn:.8f}。'
                         '请检查是否读到了另一轮概率图；本次停止导出。')

    # 对照已有 CSV，确保校准使用的是同一批测试预测。
    matched_files = []
    for name in ['submission_task4_vessels.csv', 'submission_vessel_candidate.csv', 'submission.csv']:
        path = data['root'] / name
        if not path.is_file():
            continue
        rows = t4_csv(path)
        if [r['Id'] for r in rows] != ids:
            raise ValueError(f'旧提交编号和当前测试顺序不一致：{name}')
        regions = [t4_decode(r['Predicted']) for r in rows]
        predictions = [p >= threshold for p in test512]
        same = all(np.array_equal(a, b) for a, b in zip(regions, predictions))
        complement = all(np.array_equal(a, ~b) for a, b in zip(regions, predictions))
        if not (same or complement):
            raise ValueError(f'{name} 与本次缓存的预测不对应；停止导出。')
        matched_files.append(dict(name=name, sha256=t4_hash(path), region='vessel' if same else 'background'))

    raw_strings = data['official']([str(r['mask_path']) for r in train])
    if len(raw_strings) != 20:
        raise ValueError('官方转换器没有返回 20 张标签的编码。')
    raw_targets = [t4_decode(s) for s in raw_strings]
    for target, record in zip(raw_targets, train):
        if not np.array_equal(target, t4_nonzero512(t4_gray(record['mask_path']))):
            raise ValueError('真实官方函数与预期缩放/展开规则不一致。')
    white_count = sum(r['polarity'] == 'white' for r in train)
    raw_white_is_vessel = white_count == 20 and all(.001 < y.mean() < .45 for y in raw_targets)
    if raw_white_is_vessel:
        targets, target_name = raw_targets, 'raw_labels_through_official_converter'
        methods = ['probability_then_threshold', 'native_mask_then_official_resize'] + [
            f'native_mask_then_official_resize_minarea_{area}' for area in (8, 24, 64)]
        print('原始训练标签：20/20 张为白色血管。直接按官方脚本转换原标签作为校准参照。')
    else:
        targets, target_name = nearest, 'nearest_neighbor_vessels'
        methods = ['probability_then_threshold']
        print('原标签的非零区域未能一致确认是血管；继续使用最近邻血管标签，只校准阈值。')
    print('正在用已有 OOF 概率图比较导出流程；不会训练或加载模型进行推理。', flush=True)
    candidates = [(method, float(t)) for method in methods
                  for t in sorted(set(np.round(np.arange(.10, .901, .01), 2).tolist() + [threshold]))]
    score_matrix = np.asarray([
        [t4_dice(t4_prediction(p, q, method, t), y)
         for p, q, y in zip(oof, oof512, targets)] for method, t in candidates])
    baseline_scores = np.asarray([t4_dice(p, y) for p, y in zip(baseline, targets)])
    held_out = np.zeros(20)
    fold_selection = []
    for fold in sorted(set(folds.tolist())):
        fitted, evaluated = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
        chosen = t4_select(score_matrix, candidates, fitted)
        held_out[evaluated] = score_matrix[chosen, evaluated]
        fold_selection.append(dict(fold=int(fold), method=candidates[chosen][0],
                                   threshold=candidates[chosen][1],
                                   baseline=float(baseline_scores[evaluated].mean()),
                                   calibrated=float(held_out[evaluated].mean())))
    # 将复杂后处理与同样细化阈值的简单流程比较，避免仅因搜索更多参数而选中它。
    simple_indices = [i for i, (m, _) in enumerate(candidates) if '_minarea_' not in m]
    simple_heldout = np.zeros(20)
    simple_matrix = score_matrix[simple_indices]
    simple_candidates = [candidates[i] for i in simple_indices]
    for fold in sorted(set(folds.tolist())):
        fitted, evaluated = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
        local_choice = t4_select(simple_matrix, simple_candidates, fitted)
        simple_heldout[evaluated] = simple_matrix[local_choice, evaluated]
    filtering_heldout_gain = float(held_out.mean() - simple_heldout.mean())
    filtering_accepted = filtering_heldout_gain > .001
    if not filtering_accepted:
        held_out = simple_heldout
        selected = simple_indices[t4_select(simple_matrix, simple_candidates, np.arange(20))]
        fold_selection = []
        for fold in sorted(set(folds.tolist())):
            fitted, evaluated = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
            local_choice = t4_select(simple_matrix, simple_candidates, fitted)
            fold_selection.append(dict(fold=int(fold), method=simple_candidates[local_choice][0],
                threshold=simple_candidates[local_choice][1],
                baseline=float(baseline_scores[evaluated].mean()),
                calibrated=float(simple_heldout[evaluated].mean())))
    else:
        selected = t4_select(score_matrix, candidates, np.arange(20))
    gain = float(held_out.mean() - baseline_scores.mean())
    accepted = gain > .001
    if not accepted:
        selected = candidates.index(('probability_then_threshold', threshold))
    method, selected_threshold = candidates[selected]
    predicted_oof = [t4_prediction(p, q, method, selected_threshold) for p, q in zip(oof, oof512)]
    predicted_test = [t4_prediction(p, q, method, selected_threshold) for p, q in zip(tests, test512)]

    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    mask_dir = destination / 'predicted_masks_black_vessels'
    mask_dir.mkdir(exist_ok=True)
    # 真正调用官方函数检查选定流程；临时白色图只用于血管 RLE 的编码检查。
    with tempfile.TemporaryDirectory(prefix='.encoding_check_', dir=destination) as temp:
        if not Path(temp).resolve().is_relative_to(destination.resolve()):
            raise ValueError('临时编码目录不在本次输出目录内。')
        temporary_pngs = []
        native_binary = method.startswith('native_mask_then_official_resize')
        for i, (p, region) in enumerate(zip(tests, predicted_test)):
            path = Path(temp) / f'{i:02d}.png'
            encoded_image = t4_native_binary(p, method, selected_threshold) if native_binary else region
            Image.fromarray(encoded_image.astype(np.uint8) * 255).save(path)
            temporary_pngs.append(str(path))
            # 保留任务说明所要求的黑色血管 PNG。
            Image.fromarray(np.where(region, 0, 255).astype(np.uint8)).save(mask_dir / png_names[i])
        strings = data['official'](temporary_pngs)
    if len(strings) != 20:
        raise ValueError('官方转换器输出数量不正确。')
    for text, region in zip(strings, predicted_test):
        if str(text).strip() != t4_rle(region) or not np.array_equal(t4_decode(text), region):
            raise ValueError('实际官方编码、血管像素和解码结果不一致。')
    submission = destination / 'submission_task4_calibrated.csv'
    with submission.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['Id', 'Predicted'])
        writer.writerows(zip(ids, strings))
    reread = t4_csv(submission)
    if ([r['Id'] for r in reread] != ids or not all(np.array_equal(t4_decode(row['Predicted']), region)
            for row, region in zip(reread, predicted_test))):
        raise ValueError('最终提交文件的写回校验失败。')
    changed_pixels = any(not np.array_equal(region, p >= threshold)
                         for region, p in zip(predicted_test, test512))
    export_changed = bool(accepted and changed_pixels)
    report = dict(
        previous_public_score_user_reported=None,
        separate_v3_public_score_user_reported=0.72,
        probability_source=data['source'], original_threshold=threshold,
        raw_white_vessel_labels=white_count, target_convention=target_name,
        baseline_nearest_dice=baseline_nn,
        calibrated_nearest_dice=float(np.mean([t4_dice(p, y) for p, y in zip(predicted_oof, nearest)])),
        baseline_same_target_dice=float(baseline_scores.mean()),
        selected_same_target_dice=float(score_matrix[selected].mean()),
        heldout_parameter_selection_dice=float(held_out.mean()), heldout_gain=gain,
        export_changed=export_changed, stable_oof_gain=accepted,
        simple_refined_heldout_dice=float(simple_heldout.mean()),
        filtering_heldout_gain=filtering_heldout_gain,
        filtering_accepted=filtering_accepted,
        threshold_search_step=0.01,
        selected_method=method, selected_threshold=selected_threshold,
        mean_nearest_label_fraction=float(np.mean([y.mean() for y in nearest])),
        mean_raw_official_label_fraction=float(np.mean([y.mean() for y in raw_targets])),
        mean_test_vessel_fraction=float(np.mean([y.mean() for y in predicted_test])),
        fold_parameter_selection=fold_selection, matched_old_submissions=matched_files,
        submission_file=str(submission), submission_sha256=t4_hash(submission),
        test_id_mapping=[dict(Id=sid, image=r['path'].name, mask=png_names[i])
                         for i, (sid, r) in enumerate(zip(ids, test))],
        note='隐藏评分标签生成方式未公开。参数只根据训练集 OOF 选择；OOF 曾参与模型轮次选择，'
             '分折检查只检验后处理选择稳定性，不是独立测试结果，也不保证榜分达到 0.85。')
    report_path = destination / 'calibration_report.json'
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    with (destination / 'calibration_search.csv').open('w', encoding='utf-8', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['method', 'threshold', 'mean_dice_on_same_target'])
        writer.writerows((m, t, float(score_matrix[i].mean())) for i, (m, t) in enumerate(candidates))
    print(f'原流程在同一标签定义下的 OOF Dice：{report["baseline_same_target_dice"]:.6f}')
    print(f'按折留出后处理参数后的 Dice：{report["heldout_parameter_selection_dice"]:.6f}')
    print(f'同口径改变量：{gain:+.6f}')
    print(f'最终导出流程：{method}；阈值：{selected_threshold:.2f}')
    print(f'最近邻标签 Dice（便于与此前 0.7964 对照）：{report["calibrated_nearest_dice"]:.6f}')
    if not accepted:
        print('未发现稳定改善，已保留原预测；生成文件无需重复提交。请把诊断报告发回。')
    elif not changed_pixels:
        print('校准参数未改变测试集预测，无需重复提交。请把诊断报告发回。')
    else:
        print('已生成待线上验证的校准提交；保留此前 0.72 的提交记录，实际效果以新榜分为准。')
    print('提交文件：', submission)
    print('诊断报告：', report_path)
    print('文件 SHA256：', report['submission_sha256'])
    print('本次没有训练。以下摘要可直接复制发回：')
    print(json.dumps({k: report[k] for k in ['raw_white_vessel_labels', 'target_convention',
          'baseline_same_target_dice', 'heldout_parameter_selection_dice', 'heldout_gain',
          'export_changed', 'selected_method', 'selected_threshold']}, ensure_ascii=False, indent=2))
    if show_download:
        from IPython.display import HTML, display
        files = [(report_path, '下载诊断报告')]
        if export_changed:
            files.insert(0, (submission, '下载待验证提交 CSV'))
        for path, label in files:
            mime = 'text/csv' if path.suffix == '.csv' else 'application/json'
            encoded = base64.b64encode(path.read_bytes()).decode('ascii')
            display(HTML(f'<p><a download="{path.name}" href="data:{mime};base64,{encoded}" '
                         f'style="font-size:18px;padding:10px;display:inline-block">{label}</a></p>'))
    return report


TASK4_OUTPUT_DIR = '/kaggle/working/retina_ml004_v1_calibration'
task4_calibration_report = t4_calibrate(t4_inputs(globals()), TASK4_OUTPUT_DIR, show_download=False)
chosen = (Path(task4_calibration_report['submission_file'])
          if task4_calibration_report['export_changed'] else Path(submission_path))
final_submission = ctx['out'] / 'submission_ml004_v1_recovered.csv'
final_submission.write_bytes(chosen.read_bytes())
recovery_audit.update(
    nearest_oof_dice=metrics['oof_vessel_dice_512'],
    nearest_threshold=metrics['threshold'],
    calibrated=task4_calibration_report['export_changed'],
    final_submission=str(final_submission), final_sha256=sha256_file(final_submission))
atomic_json(recovery_audit, ctx['out'] / 'recovery_audit.json')
print('\n恢复完成：只复用旧概率图，没有训练，也没有运行模型推理。')
print('本次待验证文件：', final_submission)
print('本次源模型来自Kaggle保存第1版；0.72是另一轮v3的成绩，请保留原提交记录。')
print('OOF用于后处理选择，不能保证线上达到0.85。')
print(json.dumps(recovery_audit, ensure_ascii=False, indent=2))
from IPython.display import HTML, display
for path, label in [(final_submission, '下载本次待验证CSV'),
                    (ctx['out'] / 'recovery_audit.json', '下载恢复核查记录'),
                    (Path(TASK4_OUTPUT_DIR) / 'calibration_report.json', '下载校准报告')]:
    content = base64.b64encode(path.read_bytes()).decode('ascii')
    mime = 'text/csv' if path.suffix == '.csv' else 'application/json'
    display(HTML(f'<p><a download="{path.name}" href="data:{mime};base64,{content}" '
                 f'style="font-size:18px;padding:10px;display:inline-block">{label}</a></p>'))

